# 03 — Data Splitting | 70/15/15 with real-only testing

**Input:** `citizen_feedback_raw_v1 (2).xlsx`, sheets `preprocessed_corpus` and `preprocessed_aspect_annotations`.  
**Goal:** 70% Training, 15% Validation, 15% Testing by **unique comment**, with **every Testing comment original/collected**. Target-level aspect annotations follow their parent comment.

**Current workbook snapshot:** 4,973 comments, 6,941 annotation rows; 973 original/collected comments and 4,000 synthetic ones. This results in an approximately **746-comment real-only test set**, leaving only **227 original comments** for Training + Validation. This severely limits authentic training examples; the notebook reports the exact consequences rather than implying the models have 4,973 authentic comments.

**Leakage safeguards:** source_type-based provenance; source-reference groups for original comments; exact text/hash groups; cosine-similarity near-duplicate grouping using char 3–5-gram TF-IDF at 0.90; no cross-split group leakage. Reproducible seed 42. The Testing set is reserved for final evaluation only. The updated `source_type=synthetic` field identifies generated comments; no invented platform origin is inferred.

**Methodological note:** `source_reference` can contain generic post labels or usernames, not independently verified thread IDs. Conservatively grouping identical original references avoids cross-split matches but can create large groups. Check the grouping report before treating thread isolation as proven.

## 1. Imports, paths, and split settings

Requires `pandas`, `numpy`, `matplotlib`, `scikit-learn`, and `openpyxl`. Works from the project root or from a `notebooks/` subfolder.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

SEED=42
SPLIT_ORDER=('train','validation','test')
NEAR_DUPLICATE_THRESHOLD=0.90
SYNTHETIC_MIN_ID=1132  # Historical boundary for THIS workbook, not a universal rule.
FILENAME='citizen_feedback_raw_v1 (2).xlsx'
CANDIDATES=[Path(FILENAME),Path('..')/FILENAME,Path('/mnt/data')/FILENAME]
DATA_FILE=next((p for p in CANDIDATES if p.is_file()), None)
if DATA_FILE is None:raise FileNotFoundError(f'Locate {FILENAME} and set DATA_FILE correctly.')
OUTPUT_DIR=Path('outputs')/'data_splitting'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

rng=np.random.default_rng(SEED)
print('Workbook:',DATA_FILE.resolve())
print('Outputs:',OUTPUT_DIR.resolve())

## 2. Load the two authoritative worksheets and validate their linkage

Do not use `final_annotated`, `raw`, or hidden synthetic helper sheets as training input. All annotations for the same comment stay in the same partition.

In [ ]:
comments=pd.read_excel(DATA_FILE,sheet_name='preprocessed_corpus',dtype=str,keep_default_na=False)
aspects=pd.read_excel(DATA_FILE,sheet_name='preprocessed_aspect_annotations',dtype=str,keep_default_na=False)
aspects=aspects.loc[:, ~aspects.columns.astype(str).str.startswith('Unnamed:')]
valid=lambda s:s.astype(str).str.strip().str.fullmatch(r'C\d{4,}',na=False)
comments=comments.loc[valid(comments['comment_id'])].copy().reset_index(drop=True)
aspects=aspects.loc[valid(aspects['comment_id'])].copy().reset_index(drop=True)
comments['comment_id']=comments['comment_id'].str.strip()
aspects['comment_id']=aspects['comment_id'].str.strip()
original_comment_cols=comments.columns.tolist()
original_aspect_cols=aspects.columns.tolist()

assert comments['comment_id'].is_unique,'Duplicate corpus comment IDs.'
assert aspects['aspect_id'].is_unique,'Duplicate aspect IDs.'
assert set(aspects['comment_id'])==set(comments['comment_id']),'Unmatched or unannotated comment IDs.'
assert comments['model_text'].str.strip().ne('').all(),'Blank input comment text.'
assert aspects['aspect_code'].str.fullmatch(r'A(0[1-9]|1[01])').all(),'Invalid aspect labels.'
assert aspects['sentiment'].isin(['Negative','Neutral','Positive']).all(),'Invalid sentiment values.'
matched_text=aspects['comment_id'].map(comments.set_index('comment_id')['model_text'])
assert aspects['model_text'].eq(matched_text).all(),'Aspect model_text differs from corpus text.'

# Determine real/synthetic provenance from the updated source_type field.
source_labels=comments['source_type'].astype(str).str.strip().str.casefold()
valid_sources={'facebook','tiktok','threads','x','reddit','synthetic'}
unknown_sources=sorted(set(source_labels)-valid_sources)
assert not unknown_sources,f'Unexpected or empty source_type values: {unknown_sources}'
comments['_origin']=np.where(source_labels.eq('synthetic'),'Synthetic','Original / collected')
comments['_id_number']=comments['comment_id'].str[1:].astype(int)
id_mismatches=int((comments['_id_number'].ge(SYNTHETIC_MIN_ID)!=source_labels.eq('synthetic')).sum())
if id_mismatches:
    print(f'NOTE: {id_mismatches} IDs differ from the old C1132+ cutoff; using source_type.')

# Align exported annotation metadata with authoritative preprocessed_corpus.source_type.
expected_sources=aspects['comment_id'].map(comments.set_index('comment_id')['source_type'])
source_mismatch=(aspects['source_type'].astype(str).str.strip().str.casefold()!=
                 expected_sources.astype(str).str.strip().str.casefold())
if source_mismatch.any():
    print(f'Correcting {int(source_mismatch.sum())} outdated annotation source_type values in memory (source workbook unchanged).')
    aspects.loc[source_mismatch,'source_type']=expected_sources.loc[source_mismatch]
aspects['_origin']=aspects['comment_id'].map(comments.set_index('comment_id')['_origin'])
assert aspects['_origin'].notna().all()
N=len(comments)
TARGET_SIZES={'train':int(round(.70*N)),'validation':int(round(.15*N))}
TARGET_SIZES['test']=N-TARGET_SIZES['train']-TARGET_SIZES['validation']
real_count=int(comments['_origin'].eq('Original / collected').sum())
assert real_count>=TARGET_SIZES['test'],(
    f'Cannot create real-only Testing of {TARGET_SIZES["test"]} from {real_count} original comments.'
)
print('Comments:',len(comments),'Annotations:',len(aspects))
print('Original / collected:',real_count,'Synthetic:',N-real_count)
print('Required comment split sizes:',TARGET_SIZES)
print('Real comments left for train + validation:',real_count-TARGET_SIZES['test'])
print('Annotation columns retained for model training:',original_aspect_cols)

## 3. Build leakage groups from real source references and normalized text

All comments sharing the same original `source_reference` are conservatively kept together. The synthetic references `SIMULATED_PLATFORM_NO_EXTERNAL_SOURCE` and `SYNTHETIC_GENERATED_...` are *not* treated as genuine threads and are therefore **not** grouped as shared sources. Exact text and `text_group_id` are grouped for everyone.

In [ ]:
class UnionFind:
    def __init__(self,n):
        self.parent=list(range(n))
        self.rank=[0]*n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]
            x=self.parent[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

uf=UnionFind(N)
source_seen={};hash_seen={};text_seen={}
def norm_text(x):
    return ' '.join(unicodedata.normalize('NFC',str(x)).casefold().split())
texts=[norm_text(x) for x in comments['model_text']]
for i,row in comments.iterrows():
    origin=row['_origin']
    if origin=='Original / collected':
        src=row['source_reference'].strip()
        if src and src in source_seen:uf.union(i,source_seen[src])
        elif src:source_seen[src]=i
    th=row['text_group_id'].strip()
    if th and th in hash_seen:uf.union(i,hash_seen[th])
    elif th:hash_seen[th]=i
    tt=texts[i]
    if tt in text_seen:uf.union(i,text_seen[tt])
    else:text_seen[tt]=i
print('Source/hash/exact-text groups constructed.')

## 4. Detect near duplicates using the methodology's character 3–5-gram cosine rule

TF-IDF is used here only to identify near-duplicate **groups before splitting**. No model-training features or tuned hyperparameters are fit on the held-out Testing set. Finding near-duplicate links uses the unlabeled text, not its labels. Similarity threshold: **0.90**. For changed corpora, rerun this step before generating new partitions.

In [ ]:
vectorizer=TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=1,
                           dtype=np.float32,sublinear_tf=True)
X_text=vectorizer.fit_transform(texts)
near_duplicate_edges=0
examples=[]
for start in range(0,N,64):
    similarities=X_text[start:start+64].dot(X_text.T).tocsr()
    for local_i in range(similarities.shape[0]):
        i=start+local_i
        a,b=similarities.indptr[local_i],similarities.indptr[local_i+1]
        indices=similarities.indices[a:b]
        vals=similarities.data[a:b]
        keep=(indices>i)&(vals>=NEAR_DUPLICATE_THRESHOLD)
        for j,v in zip(indices[keep],vals[keep]):
            j=int(j)
            uf.union(i,j)
            near_duplicate_edges+=1
            if len(examples)<10:examples.append((comments.at[i,'comment_id'],comments.at[j,'comment_id'],round(float(v),3)))
print('Near-duplicate pairs (cosine >=',NEAR_DUPLICATE_THRESHOLD,'):',near_duplicate_edges)
print('Example pairs:',examples)
del X_text,vectorizer,similarities

## 5. Inspect groups and require homogeneous provenance

A real/synthetic mixed group cannot be placed in a **real-only test set** without cross-split leakage. If such a group appears, stop and manually resolve the near-duplicate conflict instead of overriding provenance.

In [ ]:
group_members=defaultdict(list)
for i in range(N):group_members[uf.find(i)].append(i)
groups=list(group_members.values())
real_groups=[];synthetic_groups=[]
for group in groups:
    origins=set(comments.iloc[group]['_origin'])
    if len(origins)>1:
        ids=comments.iloc[group]['comment_id'].tolist()[:10]
        raise ValueError(f'Near-duplicate/source group mixes real and synthetic records: {ids}. Resolve before splitting.')
    (real_groups if 'Original / collected' in origins else synthetic_groups).append(group)
print('Leakage groups:',len(groups))
print('Real groups:',len(real_groups),'Synthetic groups:',len(synthetic_groups))
print('Largest real source/near-duplicate groups:',sorted([len(g) for g in real_groups],reverse=True)[:12])
print('Largest synthetic near-duplicate groups:',sorted([len(g) for g in synthetic_groups],reverse=True)[:12])
assert sum(len(g) for g in real_groups)==real_count
assert sum(len(g) for g in synthetic_groups)==N-real_count

## 6. Create balancing features (language, sentiment, aspect, platform, and code-switching)

The algorithm uses **comment-level** feature vectors. Each annotated target contributes to sentiment counts, while categories contribute one presence indicator per comment. This minimizes multi-label distribution drift subject to the hard real-only Testing constraint.

In [ ]:
codes=[f'A{i:02d}' for i in range(1,12)]
lang_names=['English','Tagalog','Ilokano','Mixed']
sentiments=['Negative','Neutral','Positive']
source_names=sorted(set(comments['source_type'].replace({'Tiktok':'TikTok'})))
feature_names=['LANG_'+x for x in lang_names]+['SW_Yes','SW_No']+[
    'ACD_'+x for x in codes]+['SENT_'+x for x in sentiments]+['SRC_'+x for x in source_names]
feature_weights=np.array([2.0]*4+[.5,.5]+[2.5]*11+[2.0]*3+[.25]*len(source_names),dtype=float)
feature_index={name:i for i,name in enumerate(feature_names)}
feat=np.zeros((N,len(feature_names)),dtype=float)
ids_to_position={c:i for i,c in enumerate(comments['comment_id'])}
for i,c in comments.iterrows():
    lang=(c.get('primary_language_canonical') or c['primary_language']).replace('Ilocano','Ilokano')
    feat[i,feature_index.get('LANG_'+lang,0)]+=1
    sw=(c.get('code_switched_canonical') or c['code_switched']).strip().title()
    if 'SW_'+sw in feature_index:feat[i,feature_index['SW_'+sw]]+=1
    src=c['source_type'].replace('Tiktok','TikTok')
    if 'SRC_'+src in feature_index:feat[i,feature_index['SRC_'+src]]+=1
for cid, part in aspects.groupby('comment_id'):
    i=ids_to_position[cid]
    for code in set(part['aspect_code']):
        feat[i,feature_index['ACD_'+code]]+=1
    for sent,n in part['sentiment'].value_counts().items():
        feat[i,feature_index['SENT_'+sent]]+=int(n)

def feature_score(actual,target):
    return float(np.sum(feature_weights*(actual-target)**2/(target+2.0)))

def random_exact_group_subset(candidate_groups,target,rng):
    # Randomized subset-sum dynamic programming; group members never split.
    order=rng.permutation(len(candidate_groups))
    dp=[None]*(target+1)
    dp[0]=()
    for gi in order:
        size=len(candidate_groups[int(gi)])
        if size>target:continue
        for j in range(target-size,-1,-1):
            if dp[j] is not None and dp[j+size] is None:
                dp[j+size]=dp[j]+(int(gi),)
    if dp[target] is None:return None
    return [candidate_groups[g] for g in dp[target]]

def flatten(gs):return [i for group in gs for i in group]
print('Balancing features:',len(feature_names))

## 7. Select the 15% Testing set — **real comments only**

First, select whole real-only leakage groups that sum to the required Testing size. Among feasible candidates, prefer representative language/aspect/sentiment distributions within the **real** subset. Once chosen, freeze these IDs; do not tune models or thresholds on Testing.

In [ ]:
TEST_TARGET=TARGET_SIZES['test']
real_positions=flatten(real_groups)
real_goal=feat[real_positions].sum(axis=0)*(TEST_TARGET/len(real_positions))
best_test=None;best_test_score=float('inf')
for attempt in range(60):
    candidate=random_exact_group_subset(real_groups,TEST_TARGET,rng)
    if candidate is None:continue
    chosen=flatten(candidate)
    sc=feature_score(feat[chosen].sum(axis=0),real_goal)
    if sc<best_test_score:
        best_test_score=sc;best_test=candidate
if best_test is None:
    raise ValueError('Exact real-only Testing size cannot be reached with current groups. Review source grouping or requested constraints.')
test_positions=set(flatten(best_test))
assert len(test_positions)==TEST_TARGET
assert comments.iloc[list(test_positions)]['_origin'].eq('Original / collected').all()
remaining_real_groups=[g for g in real_groups if g[0] not in test_positions]
print('Frozen real-only Testing comments:',len(test_positions))
print('Remaining real comments:',sum(len(g) for g in remaining_real_groups))
print('Test real-distribution balancing score:',round(best_test_score,3))

## 8. Allocate Training and Validation, respecting every leakage group

The selector combines whole multi-comment groups and singleton groups to meet the exact Validation size when feasible. It **does not assume** there are at least 746 singleton synthetic groups. If exact allocation is impossible, it raises an informative error rather than breaking duplicate groups.

In [ ]:
VAL_TARGET=TARGET_SIZES['validation']
remaining_real_positions=flatten(remaining_real_groups)
remaining_real_total=len(remaining_real_positions)
real_val_target=round(remaining_real_total*VAL_TARGET/(N-TEST_TARGET))
validation_target_features=feat[[i for i in range(N) if i not in test_positions]].sum(axis=0)*(VAL_TARGET/(N-TEST_TARGET))
real_val_goal=feat[remaining_real_positions].sum(axis=0)*real_val_target/max(remaining_real_total,1)

single_syn=[g for g in synthetic_groups if len(g)==1]
multi_syn=[g for g in synthetic_groups if len(g)>1]
syn_total=sum(len(g) for g in synthetic_groups)
multi_total=sum(len(g) for g in multi_syn)
print(f'Synthetic leakage groups: {len(single_syn)} singleton, {len(multi_syn)} multi-comment; {syn_total} comments total')
print('Largest synthetic groups:', sorted((len(g) for g in synthetic_groups), reverse=True)[:10])

def select_synthetic_validation(target, rng):
    """Pick exactly `target` synthetic comments WITHOUT breaking leakage groups.

    Unlike the earlier code, this does not require 746 singleton groups.
    It fills with whole multi-comment groups and then singletons. If an
    exact selection is impossible, a subset-sum fallback verifies that.
    """
    if target < 0 or target > syn_total:
        raise ValueError(f'Invalid synthetic validation target: {target}; total available={syn_total}')
    if target == 0:
        return []

    # Include multi-comment groups in roughly their corpus proportion, while
    # taking enough to compensate if fewer singleton groups are available.
    required_multi = max(0, target - len(single_syn))
    desired_multi = max(required_multi, round(target * multi_total / syn_total))
    selected = []
    selected_rows = 0
    group_order = rng.permutation(len(multi_syn))
    for gi in group_order:
        group = multi_syn[int(gi)]
        size = len(group)
        if selected_rows >= desired_multi:
            break
        if selected_rows + size <= target:
            selected.append(group)
            selected_rows += size

    remaining = target - selected_rows
    if 0 <= remaining <= len(single_syn):
        indices = rng.choice(len(single_syn), size=remaining, replace=False)
        return selected + [single_syn[int(i)] for i in indices]

    # Some group-size distributions make greedy selection impossible. Only
    # then use the exact, more expensive randomized subset-sum procedure.
    exact = random_exact_group_subset(synthetic_groups, target, rng)
    if exact is not None:
        return exact
    largest = sorted((len(g) for g in synthetic_groups), reverse=True)[:10]
    raise ValueError(
        f'Cannot select exactly {target} synthetic validation comments without '
        f'breaking leakage groups. Singleton groups={len(single_syn)}, '
        f'multi-comment groups={len(multi_syn)}, largest sizes={largest}. '
        'Inspect repeated text_group_id, exact-text and near-duplicate groups; '
        'do not split linked comments merely to satisfy the exact ratio.'
    )

best_val=None;best_val_score=float('inf')
real_candidates=[]
for attempt in range(40):
    r_candidate=random_exact_group_subset(remaining_real_groups,real_val_target,rng)
    if r_candidate is not None:real_candidates.append(r_candidate)
if not real_candidates:
    for offset in range(1,30):
        for t in [real_val_target-offset,real_val_target+offset]:
            if t<0 or t>=remaining_real_total:continue
            option=random_exact_group_subset(remaining_real_groups,t,rng)
            if option is not None:
                real_val_target=t;real_candidates.append(option);break
        if real_candidates:break
if not real_candidates:raise ValueError('Cannot allocate even an approximate real-validation share without splitting groups.')

for real_candidate in real_candidates:
    real_rows=flatten(real_candidate)
    synthetic_needed=VAL_TARGET-len(real_rows)
    if synthetic_needed<0:continue
    real_score=feature_score(feat[real_rows].sum(axis=0),real_val_goal)
    for _ in range(14):
        syn_candidate=select_synthetic_validation(synthetic_needed,rng)
        chosen=real_rows+flatten(syn_candidate)
        combined_score=feature_score(feat[chosen].sum(axis=0),validation_target_features)+0.25*real_score
        if combined_score<best_val_score:
            best_val_score=combined_score;best_val=(real_candidate,syn_candidate)
if best_val is None:raise RuntimeError('No feasible Validation candidate.')
val_positions=set(flatten(best_val[0])+flatten(best_val[1]))
train_positions=set(range(N))-test_positions-val_positions
assert len(val_positions)==VAL_TARGET
assert len(train_positions)==TARGET_SIZES['train']
print('Training:',len(train_positions),'Validation:',len(val_positions),'Testing:',len(test_positions))
print('Real Training:',int(comments.iloc[list(train_positions)]['_origin'].eq('Original / collected').sum()))
print('Real Validation:',int(comments.iloc[list(val_positions)]['_origin'].eq('Original / collected').sum()))
print('Real Testing:',int(comments.iloc[list(test_positions)]['_origin'].eq('Original / collected').sum()))

## 9. Final hard checks (including near-duplicate and source-reference isolation)

Fail before export if any size, provenance, group, or annotation-link condition is violated. All aspect annotations inherit the split of their parent comment; no target-level rows are independently shuffled.

In [ ]:
assignment={}
for name,indices in [('train',train_positions),('validation',val_positions),('test',test_positions)]:
    for i in indices:assignment[comments.at[i,'comment_id']]=name
assert len(assignment)==N and len(set(assignment))==N

split_of_index={i:assignment[comments.at[i,'comment_id']] for i in range(N)}
checks={
    'exact_70_15_15_sizes':all(sum(v==s for v in assignment.values())==TARGET_SIZES[s] for s in SPLIT_ORDER),
    'testing_all_original':all(comments.at[i,'_origin']=='Original / collected' for i in test_positions),
    'all_comments_assigned_once':len(assignment)==N,
    'all_annotations_linked':set(aspects['comment_id'])==set(assignment),
    'groups_not_split':all(len({split_of_index[i] for i in g})==1 for g in groups),
    'aspect_ids_unique':aspects['aspect_id'].is_unique,
    'comment_text_populated':comments['model_text'].str.strip().ne('').all(),
}
assert all(checks.values()),f'One or more split integrity checks failed: {checks}'

# Source-reference isolation is additionally rechecked for the ORIGINAL subset.
orig_ref_to_split=defaultdict(set)
for i in real_positions:
    ref=comments.at[i,'source_reference'].strip()
    if ref:orig_ref_to_split[ref].add(split_of_index[i])
checks['original_source_ref_isolated']=all(len(x)==1 for x in orig_ref_to_split.values())
checks={key:bool(value) for key,value in checks.items()}
assert all(checks.values()),checks
print('All integrity checks PASSED:',json.dumps(checks,indent=2))

## 10. Export comment splits, annotation splits, and a model-compatible Excel workbook

The three Excel sheets `Training`, `Validation`, and `Testing` contain **original aspect-annotation columns only**, so they work with the existing project's `shared/data.py`. `source_type` on exported annotation rows is synchronized with the corpus; the original input workbook is not modified. Do not overwrite the source workbook.

In [ ]:
comments_to_export=comments[original_comment_cols].copy()
aspects_to_export=aspects[original_aspect_cols].copy()
comment_to_split=comments['comment_id'].map(assignment)
aspect_to_split=aspects['comment_id'].map(assignment)

for split in SPLIT_ORDER:
    c_rows=comments_to_export.loc[comment_to_split.eq(split)].copy()
    a_rows=aspects_to_export.loc[aspect_to_split.eq(split)].copy()
    c_rows.to_csv(OUTPUT_DIR/f'{split}_comments.csv',index=False,encoding='utf-8-sig')
    a_rows.to_csv(OUTPUT_DIR/f'{split}_aspects.csv',index=False,encoding='utf-8-sig')

comments_to_export.to_csv(OUTPUT_DIR/'corpus_comments.csv',index=False,encoding='utf-8-sig')
aspects_to_export.to_csv(OUTPUT_DIR/'corpus_aspects.csv',index=False,encoding='utf-8-sig')
workbook_path=OUTPUT_DIR/'citizen_feedback_split.xlsx'
with pd.ExcelWriter(workbook_path,engine='openpyxl') as writer:
    for split,sheet in [('train','Training'),('validation','Validation'),('test','Testing')]:
        aspects_to_export.loc[aspect_to_split.eq(split)].to_excel(writer,sheet_name=sheet,index=False)

manifest=comments[['comment_id','text_group_id','source_reference','source_type','primary_language','code_switched']].copy()
manifest['split']=manifest['comment_id'].map(assignment)
manifest['data_origin']=comments['_origin']
manifest['annotation_rows']=manifest['comment_id'].map(aspects.groupby('comment_id').size())
manifest.to_csv(OUTPUT_DIR/'comment_split_manifest.csv',index=False,encoding='utf-8-sig')
print('Exported model-ready split workbook:',workbook_path.resolve())
print('Exported 6 split CSV files + full source copies + manifest.')

## 11. Split summary, subgroup balance, and audit trail

Denominators: language and provenance use unique comments, sentiment and aspect category use annotation rows. Preserve the split manifest, seed, and audit JSON with your experiments. Validation includes many synthetic records; report real-only validation subgroup scores separately when evaluating models.

In [ ]:
summary=[]
for split in SPLIT_ORDER:
    c=comments.loc[comment_to_split.eq(split)]
    a=aspects.loc[aspect_to_split.eq(split)]
    lang=c.get('primary_language_canonical',c['primary_language']).replace({'Ilocano':'Ilokano'}).value_counts()
    origin=c['_origin'].value_counts()
    sent=a['sentiment'].value_counts()
    summary.append({
        'split':split,'comments':len(c),'comment_pct':round(100*len(c)/N,2),
        'real_comments':int(origin.get('Original / collected',0)),
        'synthetic_comments':int(origin.get('Synthetic',0)),
        'aspect_rows':len(a),'Negative':int(sent.get('Negative',0)),
        'Neutral':int(sent.get('Neutral',0)),'Positive':int(sent.get('Positive',0)),
        **{lang_name:int(lang.get(lang_name,0)) for lang_name in lang_names},
    })
summary_df=pd.DataFrame(summary)
display(summary_df)
summary_df.to_csv(OUTPUT_DIR/'split_summary.csv',index=False,encoding='utf-8-sig')

all_tab=[]
for field,description,frame in [
    ('_origin','Provenance (comments)',comments),
    ('primary_language','Primary language (comments)',comments),
    ('source_type','Assigned/reported platform (comments)',comments),
    ('aspect_code','Aspect categories (annotations)',aspects),
    ('sentiment','Sentiment (annotations)',aspects),
]:
    frame_split=frame['comment_id'].map(assignment)
    series=frame[field]
    labels=series.replace({'Tiktok':'TikTok','Ilocano':'Ilokano'})
    ct=pd.crosstab(labels,frame_split).reindex(columns=list(SPLIT_ORDER),fill_value=0)
    for label,r in ct.iterrows():
        total=int(r.sum())
        all_tab.append({'family':description,'label':label,'total':total,
                        **{name:int(r[name]) for name in SPLIT_ORDER},
                        **{name+'_pct_of_label':round(100*int(r[name])/total,2) for name in SPLIT_ORDER}})
dist_df=pd.DataFrame(all_tab)
dist_df.to_csv(OUTPUT_DIR/'split_distribution_comparison.csv',index=False,encoding='utf-8-sig')
display(dist_df.head(25))

audit={
    'seed':SEED,'source_workbook':str(DATA_FILE.resolve()),
    'source_sheets':['preprocessed_corpus','preprocessed_aspect_annotations'],
    'target_ratios':{'train':0.70,'validation':0.15,'test':0.15},
    'target_comment_counts':TARGET_SIZES,
    'comment_count':N,'annotation_rows':len(aspects),
    'source_original_comments':real_count,'source_synthetic_comments':N-real_count,
    'provenance_rule':'preprocessed_corpus.source_type == synthetic indicates generated comments; social-media platforms indicate collected comments',
    'group_rule':'Real source_reference + exact normalized text + text_group_id + char 3-5 TF-IDF cosine >= 0.90',
    'near_duplicate_edges':near_duplicate_edges,
    'annotation_source_type_rows_corrected':int(source_mismatch.sum()),
    'leakage_groups':len(groups),
    'checks':checks,
    'split_summary':summary,
    'limitation':'Real-only 15% Testing retains very few original comments for Training and Validation; synthetic labels may be unreviewed.',
}
(OUTPUT_DIR/'split_config_and_checks.json').write_text(json.dumps(audit,indent=2,ensure_ascii=False),encoding='utf-8')
print('Saved audit:',OUTPUT_DIR/'split_config_and_checks.json')

## 12. Verify exported data and visualize 70/15/15 composition

This is a post-export verification. A real-only Testing set is an absolute requirement. The other partitions can contain original and synthetic comments. All results are reproducible with the same workbook, code, and seed.

In [ ]:
saved_sheets=pd.read_excel(workbook_path,sheet_name=['Training','Validation','Testing'],dtype=str,keep_default_na=False)
for split,sheet in [('train','Training'),('validation','Validation'),('test','Testing')]:
    actual_ids=set(saved_sheets[sheet]['comment_id'])
    expected_ids=set(comments.loc[comment_to_split.eq(split),'comment_id'])
    assert actual_ids==expected_ids, f'Excel export mismatch: {sheet}'
    assert len(saved_sheets[sheet])==int(aspect_to_split.eq(split).sum()),f'Annotation count mismatch: {sheet}'

assert not set(saved_sheets['Testing']['comment_id']) & set(
    comments.loc[comments['_origin'].eq('Synthetic'),'comment_id'])

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(summary_df['split'],summary_df['comments'])
for i,n in enumerate(summary_df['comments']):axes[0].text(i,n+15,str(n),ha='center')
axes[0].set(ylabel='Unique comments',title='70 / 15 / 15 split by comment')
axes[1].bar(summary_df['split'],summary_df['real_comments'],label='Original / collected')
axes[1].bar(summary_df['split'],summary_df['synthetic_comments'],bottom=summary_df['real_comments'],label='Synthetic')
axes[1].set(ylabel='Unique comments',title='Provenance within each split')
axes[1].legend(loc='upper right')
plt.tight_layout();plt.show()
print('VERIFIED: exact 70/15/15 counts, all Testing comments original/collected, no group leakage, and workbook exports consistent.')

## 13. Before model training

1. Keep the entire Testing set untouched until final evaluation. Use Training-only grouped development folds for hyperparameter selection to match your methodology.
2. The large real-only Testing partition necessarily leaves **few original comments for model development**, and Validation is largely synthetic. Report real-only Validation subgroup performance separately, or revise the research protocol if stronger authentic model selection is needed.
3. `source_reference` grouping is conservative, but identical generic labels do not independently prove posts came from the same social-media thread.
4. Review synthetic aspect/sentiment labels; auto-generated suggestions should not be represented as human-verified annotations. The `synthetic` source type must be preserved when reporting generated rows.
5. The workbook exported as `outputs/data_splitting/citizen_feedback_split.xlsx` contains the `Training`, `Validation`, and `Testing` worksheets required by your existing `shared/data.py` and GLOT500-m code.
